# Word Embedding with keras Chapter 16 Workshop 3

## Load module

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Sentences
s1 = 'CNN is great.'
s2 = 'Python is a good language.'
s3 = 'So good so happy enjoy with Python.'

# List of sentences
sentences = [s1, s2, s3]

## Tokenizer

In [5]:
tk = Tokenizer()
tk.fit_on_texts(sentences)

In [6]:
tk.document_count

3

In [7]:
tk.word_docs

defaultdict(int,
            {'great': 1,
             'is': 2,
             'cnn': 1,
             'good': 2,
             'a': 1,
             'language': 1,
             'python': 2,
             'with': 1,
             'happy': 1,
             'enjoy': 1,
             'so': 1})

In [8]:
tk.word_counts

OrderedDict([('cnn', 1),
             ('is', 2),
             ('great', 1),
             ('python', 2),
             ('a', 1),
             ('good', 2),
             ('language', 1),
             ('so', 2),
             ('happy', 1),
             ('enjoy', 1),
             ('with', 1)])

In [9]:
tk.word_index

{'is': 1,
 'python': 2,
 'good': 3,
 'so': 4,
 'cnn': 5,
 'great': 6,
 'a': 7,
 'language': 8,
 'happy': 9,
 'enjoy': 10,
 'with': 11}

In [10]:
tk.word_index['is']

1

In [11]:
sents_enc = tk.texts_to_sequences(sentences)
print(sents_enc)

[[5, 1, 6], [2, 1, 7, 3, 8], [4, 3, 4, 9, 10, 11, 2]]


In [12]:
print(s1)
tk.texts_to_sequences([s1])

CNN is great.


[[5, 1, 6]]

## Padding (Make all sentence have same shape from texts_to_sequences function)

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
# Padding sequences to ensure uniform length
max_len = 6 # Maximum length of sequences
sents_pad = pad_sequences(sents_enc, maxlen=max_len, 
                          padding='post', truncating='post')
# padding = 'post' means padding is added at the end || 'pre' would add it at the beginning.
# truncating ='post' means if the sequence is longer than max_len, cut it at the end || 'pre' would cut it at the beginning.
print(sents_pad)

[[ 5  1  6  0  0  0]
 [ 2  1  7  3  8  0]
 [ 4  3  4  9 10 11]]


### decode nomal

In [14]:
tk.sequences_to_texts(sents_enc)

['cnn is great',
 'python is a good language',
 'so good so happy enjoy with python']

### decode pad

In [15]:
tk.sequences_to_texts(sents_pad)

['cnn is great', 'python is a good language', 'so good so happy enjoy with']

## Word embedding

### Load module

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding

In [17]:
vocab_size = len(tk.word_index) + 1  # +1 for padding token
embed_len = 5
print(vocab_size)

12


### create model

In [42]:
model = Sequential()
# Paraameter size: vocab_size (number of unique words) * embed_len (embedding dimension)
# vocab_size: number of unique words in the vocabulary
# embed_len: vector length for each word
# max_len: length of the input sequences
model.add(Embedding(input_dim=vocab_size, output_dim=embed_len, input_shape=(max_len,), name='embedding_layer'))
# model.add(Embedding(input_dim=vocab_size, output_dim=embed_len, input_length=max_len, name='embedding_layer'))
# model.build((None, max_len))  # Build the model with input shape
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ (None, 6, 5)           │            60 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60 (240.00 B)

 Trainable params: 60 (240.00 B)

 Non-trainable params: 0 (0.00 B)

### predict

In [39]:
vectors = model.predict(sents_pad)
print(vectors.shape)  # Should be (number of sentences, max_len, embed_len)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
(3, 6, 5)


In [40]:
print(vectors.round(3))  # Round the output to 2 decimal places for better readability

[[[ 0.039 -0.002  0.045  0.008  0.021]
  [-0.02   0.015  0.037 -0.037 -0.038]
  [-0.002 -0.028  0.019 -0.029  0.039]
  [-0.027 -0.048  0.006 -0.015  0.046]
  [-0.027 -0.048  0.006 -0.015  0.046]
  [-0.027 -0.048  0.006 -0.015  0.046]]

 [[-0.002 -0.043 -0.03   0.044  0.017]
  [-0.02   0.015  0.037 -0.037 -0.038]
  [ 0.035  0.046  0.016  0.043  0.045]
  [-0.027 -0.05  -0.024 -0.048 -0.015]
  [ 0.034  0.018  0.001 -0.009  0.046]
  [-0.027 -0.048  0.006 -0.015  0.046]]

 [[-0.04   0.025 -0.013 -0.048 -0.031]
  [-0.027 -0.05  -0.024 -0.048 -0.015]
  [-0.04   0.025 -0.013 -0.048 -0.031]
  [ 0.039 -0.018  0.029  0.004  0.028]
  [-0.027  0.036  0.039  0.001 -0.014]
  [-0.02  -0.012 -0.017  0.016  0.001]]]


## Check vector

### 1 vector

In [32]:
print(f'"cnn" vectors')  # Shape of the embedding vector for the first sentence
print(vectors[0][0].round(3))  # Print the embedding vector for the first sentence

"cnn" vectors
[-0.    -0.    -0.044 -0.009  0.043]


### all vectors

In [22]:
print('sentence word Vector')
print('------------------------')
for i, sent in enumerate(sentences):
    print(f'{sent} : \n{vectors[i].round(3)}')  # Print each sentence and its corresponding embedding vector

sentence word Vector
------------------------
CNN is great. : 
[[-0.021 -0.024 -0.024  0.018  0.   ]
 [-0.012  0.002 -0.007 -0.001 -0.025]
 [-0.036  0.021  0.029 -0.035  0.026]
 [-0.024  0.034 -0.038  0.001  0.034]
 [-0.024  0.034 -0.038  0.001  0.034]
 [-0.024  0.034 -0.038  0.001  0.034]]
Python is a good language. : 
[[-0.005 -0.001  0.041 -0.045  0.013]
 [-0.012  0.002 -0.007 -0.001 -0.025]
 [-0.047 -0.028 -0.033  0.012 -0.033]
 [-0.034 -0.043 -0.03  -0.032  0.023]
 [-0.005 -0.024 -0.021  0.024 -0.047]
 [-0.024  0.034 -0.038  0.001  0.034]]
So good so happy enjoy with Python. : 
[[-0.015 -0.045 -0.05   0.003 -0.012]
 [-0.034 -0.043 -0.03  -0.032  0.023]
 [-0.015 -0.045 -0.05   0.003 -0.012]
 [-0.002  0.019  0.005  0.034  0.032]
 [ 0.009 -0.035 -0.023 -0.024  0.011]
 [ 0.006 -0.006 -0.038  0.017 -0.05 ]]


In [23]:
# print(tk.word_index)
print(vectors.shape)

(3, 6, 5)


In [24]:
# vectors: embedding vectors for all sentences

# sents: vectors in sentences
# i: sentence index

# words: words in sentences

# j: word index in sentences (words)
# word_v: embedding vector for the word
for i, sents in enumerate(vectors):
    # print(f'Sentence {i+1} vectors:')
    print(sents.round(3))  # Print the embedding vectors for each sentence
    words = tk.sequences_to_texts(sents_pad)[i].split()
    for j, word_v in enumerate(sents):
        if j < len(words):
            print(f'Word: {words[j]}, Vector: {word_v.round(3)}')
        else:
            print(f'Padding Vector: {word_v.round(3)}')

[[-0.021 -0.024 -0.024  0.018  0.   ]
 [-0.012  0.002 -0.007 -0.001 -0.025]
 [-0.036  0.021  0.029 -0.035  0.026]
 [-0.024  0.034 -0.038  0.001  0.034]
 [-0.024  0.034 -0.038  0.001  0.034]
 [-0.024  0.034 -0.038  0.001  0.034]]
Word: cnn, Vector: [-0.021 -0.024 -0.024  0.018  0.   ]
Word: is, Vector: [-0.012  0.002 -0.007 -0.001 -0.025]
Word: great, Vector: [-0.036  0.021  0.029 -0.035  0.026]
Padding Vector: [-0.024  0.034 -0.038  0.001  0.034]
Padding Vector: [-0.024  0.034 -0.038  0.001  0.034]
Padding Vector: [-0.024  0.034 -0.038  0.001  0.034]
[[-0.005 -0.001  0.041 -0.045  0.013]
 [-0.012  0.002 -0.007 -0.001 -0.025]
 [-0.047 -0.028 -0.033  0.012 -0.033]
 [-0.034 -0.043 -0.03  -0.032  0.023]
 [-0.005 -0.024 -0.021  0.024 -0.047]
 [-0.024  0.034 -0.038  0.001  0.034]]
Word: python, Vector: [-0.005 -0.001  0.041 -0.045  0.013]
Word: is, Vector: [-0.012  0.002 -0.007 -0.001 -0.025]
Word: a, Vector: [-0.047 -0.028 -0.033  0.012 -0.033]
Word: good, Vector: [-0.034 -0.043 -0.03  -0.0